In [ ]:
# importing libraries

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('final_cohort.csv')

In [ ]:
df.head()

In [ ]:
# Reading in the demographics to add in race and education

demo = pd.read_csv('ClinicalV1_anonymized.csv')

In [ ]:
# keep only the id + the two extra columns, then merge on sample_id

df = df.merge(
    demo[['sample_id', 'race', 'years_of_education']],
    on='sample_id', how='left'
)

In [ ]:
# Map the numeric codes to labels
sex_map  = {-1: 'Not Stated', 1: 'Male', 2: 'Female'}   
race_map = {-1: 'Not Stated' , 1: 'American_Indian/Alaska_Native', 2: 'Black or AfricanAmerican',
            3: 'Native Hawaiian or Other Pacific Islander', 4: 'Asian', 5: 'White' , 66: 'Other' ,
            77: 'Non Caucasian or Black', 88: 'Non Caucasian'}  
df['sex_label']  = df['sex'].map(sex_map)
df['race_label'] = df['race'].map(race_map)

In [ ]:
# Label train/test
# Reproducing the split use in the models with the samewith the same seed

train_ids, test_ids = train_test_split(df['sample_id'], test_size=0.2,
                                         stratify=df['ad'], random_state=42)


In [ ]:
# Assigns rows with whether they are train or test
df['split'] = np.where(df['sample_id'].isin(train_ids), 'train', 'test')

In [ ]:
# Changing continuous variables to NaN when it is less than 0

df.loc[df['years_of_education'] < 0, 'years_of_education'] = np.nan
df.loc[df['age_at_visit'] < 0, 'age_at_visit'] = np.nan

In [ ]:
# Function that presents the summary statistics needed- can then be run on all train and test
def summarise(data, label):
    # Numbers in data
    print(f"\n= {label} (n={len(data)})")
    a = data['age_at_visit']
    # Age statistics
    print(f"Age: median {a.median():.1f}, mean {a.mean():.1f}, SD {a.std():.1f}")
    # Education statistics
    print(f"Education (years): median {data['years_of_education'].median():.1f}, "
          f"mean {data['years_of_education'].mean():.1f}, SD {data['years_of_education'].std():.1f}")
    # For loop that describes sex and race
    for col in ['sex_label', 'race_label']:
        print(f"\n{col}:")
        counts = data[col].value_counts(dropna=False) # calculates counts
        pct    = data[col].value_counts(dropna=False, normalize=True) * 100 # calculates percentages
        for cat in counts.index:
            print(f"  {cat}: {counts[cat]} ({pct[cat]:.1f}%)")

In [ ]:
# For full cohort
summarise(df, 'Full cohort')

In [ ]:
# For training cohort
summarise(df[df['split']=='train'], "Train")

In [ ]:
# For test cohort
summarise(df[df['split']=='test'],  "Test")

In [ ]:
# Summarise just AD patients

summarise(df[df['ad']==1],  "AD")

In [ ]:
# Summarise just controls

summarise(df[df['ad']==0],  "Controls")